# Amazon ESCI × OpenSearch: 検索品質の診断と改善案

このノートブックは `shopping_queries_dataset_examples` から日本語クエリを30件、失敗要因が偏らないように層化して抽出し、全339,059商品を投入した `amazon-jp` インデックスへ実際に検索を行った記録です。

評価の目的は、(1) データ投入・日本語解析器、(2) lexical ranking、(3) semantic retrieval、(4) learning-to-rank のどこに改善余地があるかを分離することです。実行してもインデックスは変更しません。

In [1]:
from pathlib import Path
import json, math, re, unicodedata
import numpy as np
import pandas as pd
import requests
from IPython.display import display, Markdown

BASE_URL = "http://localhost:9200"
INDEX_NAME = "amazon-jp"
DATA_DIR = Path("esci-data/shopping_queries_dataset")
EXAMPLES_PATH = DATA_DIR / "shopping_queries_dataset_examples.parquet"
PRODUCTS_PATH = DATA_DIR / "shopping_queries_dataset_products.parquet"
TIMEOUT = 30

session = requests.Session()
root = session.get(BASE_URL, timeout=TIMEOUT); root.raise_for_status()
count = session.get(f"{BASE_URL}/{INDEX_NAME}/_count", timeout=TIMEOUT); count.raise_for_status()
print(f"OpenSearch: {root.json()['version']['number']} / index={INDEX_NAME} / docs={count.json()['count']:,}")

OpenSearch: 3.7.0 / index=amazon-jp / docs=339,059


## 1. データと索引カバレッジ

ESCIラベルは `E=Exact`, `S=Substitute`, `C=Complement`, `I=Irrelevant`。ランキング評価では E/S/C/I を 3/2/1/0 のgainに変換します。なおESCIは検索結果全体を網羅した relevance judgment ではなく候補商品へのラベルなので、検索結果中の未判定商品 `U` を I と断定しません。

In [2]:
examples = pd.read_parquet(EXAMPLES_PATH, filters=[("product_locale", "=", "jp")])
products = pd.read_parquet(PRODUCTS_PATH, filters=[("product_locale", "=", "jp")])
mapping = session.get(f"{BASE_URL}/{INDEX_NAME}/_mapping", timeout=TIMEOUT).json()[INDEX_NAME]["mappings"]["properties"]
coverage_summary = pd.DataFrame([{
    "ESCI日本語商品数": products.product_id.nunique(),
    "OpenSearch文書数": count.json()["count"],
    "概算投入率": count.json()["count"] / products.product_id.nunique(),
    "日本語analyzer明示フィールド数": sum("analyzer" in v for v in mapping.values()),
}])
coverage_display = coverage_summary.copy()
coverage_display["概算投入率"] = coverage_display["概算投入率"].map(lambda x: f"{x:.1%}")
display(coverage_display)
print("mapped fields:", ", ".join(mapping))

,ESCI日本語商品数,OpenSearch文書数,概算投入率,日本語analyzer明示フィールド数
0,339059,339059,100.0%,0


mapped fields: product_brand, product_bullet_point, product_color, product_description, product_id, product_locale, product_title


**読み方:** 投入率が100%未満なら、モデル以前に relevant product が検索対象外です。また現mappingは `text` に日本語analyzerが明示されておらず、標準解析器のままです。Kuromojiを入れたコンテナを使っていても、mappingで指定しなければ日本語形態素解析は適用されません。

## 2. 30クエリの層化サンプル

乱数seedだけの30件では同種のクエリに偏りやすいため、代表的な難しさを含む固定の層化診断サンプルにしました。すべて `shopping_queries_dataset_examples` に実在する `query_id` です。将来比較時にも同じ集合を再利用できます。

In [3]:
sample_specs = {
    82972:  "属性・否定条件（A4/テープなし）",
    92648:  "綴り誤り/未知ブランドらしき英字",
    51965:  "英語ブランド＋カテゴリ",
    119872: "日本語の明確なカテゴリ",
    123957: "日本語複合語",
    125063: "広いカテゴリ語",
    129196: "複合意図・カテゴリ境界",
    58451:  "長い商品名風クエリ",
    120267: "ブランド＋適合条件",
    127228: "作品名＋商品タイプ",
    122801: "曖昧な単語",
    124760: "ゲーム作品名",
    123338: "用途・課題解決",
    129890: "健康カテゴリ＋成分",
    54841:  "型番＋アクセサリ",
    117598: "用途＋カテゴリ＋対象者",
    122259: "多義語",
    116:    "記号を含むブランド",
    114017: "英語作品名＋型番",
    124889: "専門カテゴリ語",
    125672: "固有商品名",
    126907: "作品タイトル",
    129244: "人物・作品エンティティ",
    127927: "短いニッチカテゴリ",
    121931: "広い素材・食品語",
    102534: "ブランド表記ゆれ",
    122508: "ブランド＋交換品",
    123455: "高級ブランド＋カテゴリ",
    7680:   "非常に長い商品名風クエリ",
    41495:  "英語の広いカテゴリ語",
}
sample = (examples[examples.query_id.isin(sample_specs)]
          .groupby(["query_id", "query", "split"], as_index=False)
          .agg(judged=("product_id", "size"), exact=("esci_label", lambda s: (s == "E").sum())))
sample["query_type"] = sample.query_id.map(sample_specs)
display(sample[["query_id", "query", "query_type", "judged", "exact"]])

,query_id,query,query_type,judged,exact
0,116,&ハニー シャンプー,記号を含むブランド,40,19
1,7680,akarinメルセデスベンツ フェードなしのウェルカムライト、ドアランプ ロゴ カーテシラン...,非常に長い商品名風クエリ,40,3
2,41495,flower,英語の広いカテゴリ語,16,15
3,51965,hugo boss shoes,英語ブランド＋カテゴリ,14,12
4,54841,iqos3 ケース,型番＋アクセサリ,16,15
5,58451,kisstyle 自転車 スポーティ サドル お尻にやさしい クッション 穴あき 圧迫感軽減...,長い商品名風クエリ,40,1
6,82972,pp袋 a4 テープなし,属性・否定条件（A4/テープなし）,40,24
7,92648,shkkn kelly,綴り誤り/未知ブランドらしき英字,16,15
8,102534,thefaceshop,ブランド表記ゆれ,16,10
9,114017,wrc 8: fia world rally championship,英語作品名＋型番,13,9


## 3. OpenSearchへ実投下してオフライン評価

比較する検索器は、チュートリアル相当の title-only `match` と、title/brand/bullet/description を重み付きで使う `multi_match` です。これはモデル投入前に確認すべき、最小のlexical baselineです。

In [4]:
GAIN = {"E": 3, "S": 2, "C": 1, "I": 0}

def mget_existing(ids):
    r = session.post(f"{BASE_URL}/{INDEX_NAME}/_mget", json={"ids": list(ids)}, timeout=TIMEOUT)
    r.raise_for_status()
    return {d["_id"] for d in r.json()["docs"] if d.get("found")}

def search(query, variant="title", size=50):
    if variant == "title":
        q = {"match": {"product_title": {"query": query, "operator": "or"}}}
    else:
        q = {"multi_match": {"query": query, "type": "best_fields", "operator": "or",
             "fields": ["product_title^4", "product_brand^2", "product_bullet_point", "product_description^0.5"]}}
    body = {"size": size, "_source": ["product_id", "product_title", "product_brand"], "query": q}
    r = session.post(f"{BASE_URL}/{INDEX_NAME}/_search", json=body, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()["hits"]["hits"]

def dcg(gains):
    return sum((2**g - 1) / math.log2(i + 2) for i, g in enumerate(gains))

def metrics(hits, labels, indexed_ids, k=10):
    gains = [GAIN.get(labels.get(h["_id"], "U"), 0) for h in hits[:k]]
    ideal = sorted((GAIN[labels[x]] for x in indexed_ids), reverse=True)[:k]
    relevant = {x for x in indexed_ids if labels[x] in {"E", "S"}}
    found50 = {h["_id"] for h in hits[:50]}
    first = next((i + 1 for i, h in enumerate(hits[:k]) if labels.get(h["_id"]) in {"E", "S"}), None)
    return {
        "nDCG@10": dcg(gains) / dcg(ideal) if dcg(ideal) else np.nan,
        "Recall@50(indexed E/S)": len(found50 & relevant) / len(relevant) if relevant else np.nan,
        "MRR@10": 1 / first if first else 0.0,
        "judged@10": sum(h["_id"] in labels for h in hits[:k]) / k,
    }

rows, hit_cache = [], {}
for rec in sample.itertuples():
    g = examples[examples.query_id == rec.query_id]
    labels = dict(zip(g.product_id, g.esci_label))
    indexed = mget_existing(g.product_id)
    total_rel = sum(x in {"E", "S"} for x in labels.values())
    indexed_rel = sum(labels[x] in {"E", "S"} for x in indexed)
    for variant in ["title", "multi_field"]:
        hits = search(rec.query, variant)
        hit_cache[(rec.query_id, variant)] = hits
        rows.append({"query_id": rec.query_id, "query": rec.query, "variant": variant,
                     "judged products": len(g), "indexed judged": len(indexed),
                     "relevant index coverage": indexed_rel / total_rel if total_rel else np.nan,
                     **metrics(hits, labels, indexed)})
results = pd.DataFrame(rows)
display(results.round({c: 3 for c in ["relevant index coverage", "nDCG@10", "Recall@50(indexed E/S)", "MRR@10", "judged@10"]}))

,query_id,query,variant,judged products,indexed judged,relevant index coverage,nDCG@10,Recall@50(indexed E/S),MRR@10,judged@10
0,116,&ハニー シャンプー,title,40,40,1.0,0.299,0.103,1.000,0.3
1,116,&ハニー シャンプー,multi_field,40,40,1.0,0.299,0.103,1.000,0.3
2,7680,akarinメルセデスベンツ フェードなしのウェルカムライト、ドアランプ ロゴ カーテシラン...,title,40,40,1.0,0.495,0.875,1.000,0.6
3,7680,akarinメルセデスベンツ フェードなしのウェルカムライト、ドアランプ ロゴ カーテシラン...,multi_field,40,40,1.0,0.495,0.875,1.000,0.6
4,41495,flower,title,16,16,1.0,0.716,0.667,1.000,0.6
5,41495,flower,multi_field,16,16,1.0,0.716,0.667,1.000,0.6
6,51965,hugo boss shoes,title,14,14,1.0,0.727,0.846,1.000,0.6
7,51965,hugo boss shoes,multi_field,14,14,1.0,0.727,0.846,1.000,0.6
8,54841,iqos3 ケース,title,16,16,1.0,0.870,0.625,1.000,0.8
9,54841,iqos3 ケース,multi_field,16,16,1.0,0.870,0.625,1.000,0.8


In [5]:
metric_cols = ["nDCG@10", "Recall@50(indexed E/S)", "MRR@10", "judged@10"]
summary = results.groupby("variant")[metric_cols].mean()
display(summary.round(3))
display(Markdown("**注意:** `judged@10` が低いほど、未判定商品を0 gainとして扱う nDCG は過小評価されます。モデル比較には同じcandidate/judgment poolを使うか、追加判定が必要です。"))

,nDCG@10,Recall@50(indexed E/S),MRR@10,judged@10
variant,,,,
multi_field,0.499,0.496,0.747,0.493
title,0.477,0.460,0.736,0.467


**注意:** `judged@10` が低いほど、未判定商品を0 gainとして扱う nDCG は過小評価されます。モデル比較には同じcandidate/judgment poolを使うか、追加判定が必要です。

### 全件投入後・30クエリ評価の要約（2026-06-20）

- 索引カバレッジは **100%（339,059 / 339,059）**。全30クエリでESCI判定対象商品も100%索引内にあり、以降の失敗は投入欠損ではない。
- title-only → multi-fieldで平均 **nDCG@10: 0.477 → 0.499**、**Recall@50: 0.460 → 0.496**、**MRR@10: 0.736 → 0.747**。改善はあるが小さく、フィールド追加だけでは限界がある。
- `shkkn kelly` はtitle-onlyでnDCG@10=0、multi-fieldで **0.641**、Recall@50も **0 → 0.938**。未知語・綴り揺れでは補助フィールドが候補回収に効いた。
- `ペットマーキング防止スプレー`、`ミニターボファン`、`ワイヤレス充電器` は全件投入後も **nDCG@10=0**。日本語解析、複合語、用途理解を改善すべき代表例。
- `自転車スピーカー` はnDCG@10=0.168で、上位に警報器やベル（I）が並ぶ。商品タイプ・用途を捉えるdense retrievalまたはcross-encoder/LTRが有望。
- `花譜`（0.095）、`フラミンゴ`（0.178）など短い固有名・曖昧語も弱い。一方、`ボッテガヴェネタ 財布`は1.000で、明確なブランド＋カテゴリは現BM25でも強い。
- judged@10は平均約0.49。未判定 `U` はnegativeと断定せず、LTR学習ではESCI candidate内のpair、または追加判定済みpoolを使う。

## 4. 上位結果を目視して失敗を分類

`U` は未判定です（irrelevantではありません）。E/S/C/I が付いた商品を優先して、上位5件を確認します。

In [6]:
inspection = []
for rec in sample.itertuples():
    g = examples[examples.query_id == rec.query_id]
    labels = dict(zip(g.product_id, g.esci_label))
    for rank, h in enumerate(hit_cache[(rec.query_id, "multi_field")][:5], 1):
        inspection.append({"query": rec.query, "rank": rank, "label": labels.get(h["_id"], "U"),
                           "score": round(h["_score"], 3), "product_id": h["_id"],
                           "title": h["_source"].get("product_title", "")[:100]})
inspection = pd.DataFrame(inspection)
pd.set_option("display.max_colwidth", 110)
display(inspection)

,query,rank,label,score,product_id,title
0,&ハニー シャンプー,1,S,33.047,B07G1HMYW4,ラックス ルミニーク ミステリアスドリーム ノンシリコン シャンプー + トリートメント（ハニー&ミステリアススパイスの香り） 450g + 450g
1,&ハニー シャンプー,2,E,31.659,B079TLLKPS,花蜜精 みつばちハニー 3点セット シャンプー コンディショナー ボディソープ 500ml
2,&ハニー シャンプー,3,U,26.116,B00005HEMA,ハニー・ビー
3,&ハニー シャンプー,4,U,26.116,B00RE7I5TU,ハニー・フラッパーズ
4,&ハニー シャンプー,5,U,25.383,B001TKMDTG,ハニー キャラメルシュガー 1kg
...,...,...,...,...,...,...
140,鉄分 サプリメント,1,E,40.903,B01N1SHREO,鉄分 サプリメント noi ヘム鉄 鉄 サプリ 日本製 国産 鉄分補給
141,鉄分 サプリメント,2,S,35.316,B01LZTXGRM,さくらの森 美めぐり習慣 ヘム鉄サプリメント 150粒 約1か月分 鉄分サプリメント アセロラ モロヘイヤ オーガニック生姜エキス配合
142,鉄分 サプリメント,3,E,31.594,B081J44TNW,【 満足度ナンバー１ 】 超濃縮 鉄分 サプリメント 【 メガ鉄 】ヘム鉄 ビタミンC B6 B12 葉酸 プルーンエキス 配合/国内製造品
143,鉄分 サプリメント,4,E,30.760,B07GGW87QS,CELEBLISSTA ( セレブリスタ ) サプリメント ヘム鉄 30日分 ( 90粒 / 3粒あたり9mg配合 ) 基礎サプリ 鉄分補給 ビタミン ミネラル配合 ( 日本製 )


## 5. 問題別の改善仮説

|問題クラス|このサンプルで見るクエリ|主因|最初に試す改善|期待するモデル|
|---|---|---|---|---|
|短い固有名・曖昧語|花譜、フラミンゴ、パイプ|意図・カテゴリが一語から確定しない|カテゴリ推定と人気度、semantic候補を追加|intent classifier + LTR|
|日本語の分かち書き/複合語|ミニターボファン、ワイヤレス充電器|default analyzerで語境界が弱い|Kuromoji + readingform、文字2–3gram subfieldで再索引|BM25を強い第一段にする|
|制約・適合・否定|PP袋 A4 テープなし、スズキ スマートキーケース|BM25は「テープ付」「他社用」も上げる|数値・ブランド・適合・否定一致を特徴量化|LambdaMART LTR / cross-encoder|
|typo/表記ゆれ|shkkn kelly|転記ミス、ローマ字、未知語|char n-gram、fuzzy、query correction|文字モデルまたはmultilingual encoder|
|語彙差・長文意図|自転車スピーカー、長いサドルquery|同義表現と意味的近さをBM25だけで捉えにくい|BM25 + dense kNNをRRFで融合|multilingual bi-encoder|
|曖昧/広い意図|広いカテゴリ語|商品タイプ・用途が混在|カテゴリ/ブランド推定、行動ログ特徴|LTR + intent classifier|
|上位の精密化|全般|候補はあるが順序が弱い|top 50–100だけquery-productを再採点|multilingual cross-encoder|

## 6. 推奨アーキテクチャと学習方法

### 優先順位 0: 測定と索引を直す

1. 全339,059件の投入は完了。今後も `_bulk.items[].error` と期待件数を検査し、投入完全性を回帰テストにする。
2. Kuromojiの `search` analyzer、英数字を壊さないkeyword/whitespace系subfield、文字2–3gram subfieldを持つ新indexへreindexする。title/brand/category/attributesを分離する。
3. 時系列またはquery_id単位でtrain/validation/testを分離し、nDCG@10、Recall@100、ゼロ件率、latencyを固定評価する。未判定=Iとはしない。

### 優先順位 1: Learning to Rank（最も説明しやすい改善）

- **モデル:** LambdaMART（LightGBM ranker等）。query単位のgroupを保ち、ESCI gain E/S/C/I=3/2/1/0でpairwise/listwise学習。
- **候補:** 改善BM25とdense検索の和集合 top 100–300。学習時も本番candidate generatorで採ったhard negativesを使う。
- **lexical特徴:** title/brand/bullet/description別BM25、phrase一致、文字ngram一致、query coverage、編集距離、数値・型番の完全一致。
- **構造特徴:** brand/category/color/size/compatibility一致、欠落・矛盾フラグ、query/title長。
- **semantic特徴:** bi-encoder cosine、cross-encoder score。ログがあればCTR/CVR・人気度はposition biasを補正して追加。
- **リーク防止:** 同一query_idをtrain/testに跨がせない。product popularityは評価期間より過去だけから作る。

### 優先順位 2: Hybrid semantic search

- **bi-encoder:** 日本語を含むmultilingual sentence embeddingをquery/productでdual-encoder化。商品文は `title [SEP] brand [SEP] bullet` とし、descriptionは切り詰める。
- **学習:** まず既成multilingual encoder、次にEをpositive、Sを弱positive、C/IとBM25上位誤りをhard negativeとしてcontrastive fine-tuning。
- **検索:** OpenSearch k-NN(HNSW) top 100とBM25 top 100をRRFで融合。dense単独化は型番・ブランド完全一致を壊しやすい。
- **再ランキング:** traffic/latencyに余裕があれば融合top 50へmultilingual cross-encoder。ESCIをgain付きclassificationまたはpairwise lossで微調整する。

### 実験順序

`完全投入 → Kuromoji/char-ngram BM25 → LambdaMART → BM25+dense RRF → cross-encoder` の順でablationを取り、各段の増分とp95 latencyを記録するのが安全です。最初からdense検索だけに置換しないことを推奨します。

## 7. 次の実装で保存する学習テーブル

1行を `(query_id, product_id)` とし、`split, label_gain, lexical scores, structured matches, semantic score, rank positions` をParquetへ保存します。これによりモデルを替えてもcandidate generationとfeature generationを再現できます。今回の30件は診断用スモークテスト、正式判断はESCI test全体または十分な固定query集合で行います。